# Feature Engineering — Pneumonia Chest X-Ray Classification

## Background

This notebook is Phase 1 of the capstone methodology: **manual feature engineering** on
the chest X-ray dataset, before any classifier is trained. Instead of feeding raw pixels
straight into a model, we first derive a small set of interpretable, hand-crafted features
per image — summarizing its pixel intensity distribution, its texture (via a gray-level
co-occurrence matrix), its edge structure, and its left-right regional symmetry. These
features become the tabular dataset used by the classical-ML baseline and primary models
in the next phases (`02_eda.ipynb`, `03_ml_modeling.ipynb`), and they are directly
comparable — on the same `test` split — against the transfer-learning model trained
straight on pixels in `04_dl_cv_transfer_learning.ipynb`.

The four feature categories, implemented in `lab/src/features.py`, are:

- **Pixel statistics** — mean, standard deviation, min, and max grayscale intensity.
- **Texture** — contrast, homogeneity, energy, and entropy computed from a gray-level
  co-occurrence matrix (GLCM) quantized to 8 gray levels. A benchmark on 300 real training
  images (2026-09-03) found GLCM timing statistically indistinguishable between 8 and 256
  gray levels at this image size — quantizing to a small level count is about **feature
  stability** (a less sparse co-occurrence matrix per image), not runtime.
- **Edge structure** — edge pixel density and contour count from Canny edge detection.
- **Region symmetry** — the skewness of the intensity distribution and a left-right
  mirror-symmetry score, motivated by the fact that pneumonia often presents as localized,
  asymmetric opacity rather than a uniformly distributed pattern.

Every image is resized to `IMG_SIZE=224` (matching the resize used for the deep-learning
track, so the two phases start from the same spatial resolution) before features are
extracted. The same benchmark measured ~6ms/image for the full extraction pipeline,
extrapolating to ~35 seconds for the full 5856-image dataset — cheap enough to run
sequentially, single-threaded, with no need for multiprocessing.

### Imports and setup

`lab/src/features.py` holds the feature-extraction functions; it is not an installed
package, so we add `lab/src/` to `sys.path` before importing from it, the same convention
already used by the other notebooks in this project.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, "../src")

from features import build_feature_table

### Building the training feature table

The `train` split is the largest of the three (1341 NORMAL + 3875 PNEUMONIA = 5216 images)
and is processed first, since it is what the training pool for Phase 3's cross-validation
will draw from.

In [2]:
train_df = build_feature_table(Path("../data/train"))
train_df.shape

(5216, 15)

### Building the validation feature table

The `val` split (8 NORMAL + 8 PNEUMONIA = 16 images) is small — it is used by Phase 4's
early stopping, not by Phase 3's cross-validation directly, but it is included here so a
feature table exists for every split of the dataset.

In [3]:
val_df = build_feature_table(Path("../data/val"))
val_df.shape

(16, 15)

### Building the test feature table

The `test` split (234 NORMAL + 390 PNEUMONIA = 624 images) is held out entirely from
training and cross-validation; its feature table is what Phase 3's final model evaluation
and Phase 5's model comparison will read.

In [4]:
test_df = build_feature_table(Path("../data/test"))
test_df.shape

(624, 15)

### Data-quality checks

Before persisting the feature tables, we confirm two things: no feature is missing or
non-finite for any image (a `NaN` here would silently corrupt every downstream model), and
the row counts match the verified dataset counts (train=5216, val=16, test=624) — a
mismatch would mean the directory walk in `build_feature_table` skipped or duplicated
images.

In [5]:
splits = {"train": train_df, "val": val_df, "test": test_df}
expected_counts = {"train": 5216, "val": 16, "test": 624}

for name, df in splits.items():
    print(f"--- {name} ---")
    print("shape:", df.shape)
    print("missing values per column:")
    print(df.isna().sum())
    print()
    assert len(df) == expected_counts[name], (
        f"{name}: expected {expected_counts[name]} rows, got {len(df)}"
    )

train_df.describe()

--- train ---
shape: (5216, 15)
missing values per column:
Image_ID           0
File_Path          0
Mean_Intensity     0
Std_Intensity      0
Min_Intensity      0
Max_Intensity      0
Contrast           0
Homogeneity        0
Energy             0
Entropy            0
Edge_Density       0
Contour_Count      0
Brightness_Dist    0
Symmetry           0
Label              0
dtype: int64

--- val ---
shape: (16, 15)
missing values per column:
Image_ID           0
File_Path          0
Mean_Intensity     0
Std_Intensity      0
Min_Intensity      0
Max_Intensity      0
Contrast           0
Homogeneity        0
Energy             0
Entropy            0
Edge_Density       0
Contour_Count      0
Brightness_Dist    0
Symmetry           0
Label              0
dtype: int64

--- test ---
shape: (624, 15)
missing values per column:
Image_ID           0
File_Path          0
Mean_Intensity     0
Std_Intensity      0
Min_Intensity      0
Max_Intensity      0
Contrast           0
Homogeneity        0
Ene

,Mean_Intensity,Std_Intensity,Min_Intensity,Max_Intensity,Contrast,Homogeneity,Energy,Entropy,Edge_Density,Contour_Count,Brightness_Dist,Symmetry
count,5216.000000,5216.000000,5216.000000,5216.000000,5216.000000,5216.000000,5216.000000,5216.000000,5216.000000,5216.000000,5216.000000,5216.000000
mean,122.990683,56.645527,0.977377,237.797929,0.169898,0.927626,0.382683,3.271964,0.051413,52.794287,-0.579232,0.888357
std,18.536930,9.503225,4.710555,19.146641,0.039606,0.014341,0.047098,0.292707,0.022291,21.348554,0.406771,0.031115
min,60.688576,20.034699,0.000000,151.000000,0.053491,0.871813,0.285637,1.820008,0.000817,2.000000,-2.759822,0.730924
25%,111.891935,50.306472,0.000000,224.000000,0.139469,0.916991,0.349110,3.098444,0.034195,38.000000,-0.780808,0.870217
50%,122.849081,57.069798,0.000000,246.000000,0.168812,0.928815,0.375629,3.290105,0.048848,50.000000,-0.557101,0.892004
75%,134.476996,63.424411,0.000000,255.000000,0.197394,0.938313,0.405449,3.498317,0.067841,67.000000,-0.347940,0.910086
max,221.533781,87.324622,142.000000,255.000000,0.394038,0.973272,0.695213,4.019305,0.131936,148.000000,0.904190,0.958454


### Saving the feature tables

Persisting the three feature tables as CSVs means `02_eda.ipynb` and `03_ml_modeling.ipynb`
can load a small, versioned tabular dataset directly instead of re-running the ~35-second
extraction pipeline every time. `lab/data/features/` is created if it does not already
exist.

In [6]:
features_dir = Path("../data/features")
features_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(features_dir / "train.csv", index=False)
val_df.to_csv(features_dir / "val.csv", index=False)
test_df.to_csv(features_dir / "test.csv", index=False)

print(
    f"Saved {len(train_df)} train rows, {len(val_df)} val rows, "
    f"{len(test_df)} test rows to {features_dir}/"
)

Saved 5216 train rows, 16 val rows, 624 test rows to ../data/features/


### Conclusion

The manual feature-engineering pipeline produced a structured, tabular dataset from the
raw chest X-ray images: 5216 training rows, 16 validation rows, and 624 test rows, each
with 12 numeric features (4 pixel-statistics, 4 texture, 2 edge, 2 region) plus
`Image_ID`, `File_Path`, and `Label`. All feature values are finite and no rows are
missing, so the three CSVs under `lab/data/features/` are ready to be consumed directly by
`02_eda.ipynb` for exploratory analysis and by `03_ml_modeling.ipynb` for the classical-ML
baseline and primary models.